# TaskMate — Validation Notebook 05: Agent Loop (Nemotron)

**Purpose:** Verify the iterative agent loop (`TaskMateAgent`) orchestrating intent recognition, structured tool calling, observation handling, and truthful response synthesis.

**Operational Note:** This notebook is strictly for isolated experimentation, learning, and verification. It is **NOT** the production runtime.

## 1. Setup Environment and Imports

In [ ]:
import os
import sys
import json

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from backend.app.agent.agent import TaskMateAgent
from backend.app.agent.tool_registry import ToolRegistry
from backend.app.core.config import get_settings
from backend.app.services.llm_service import LLMService

settings = get_settings()
print(f"Active LLM Provider: {settings.LLM_PROVIDER}")
print(f"Active LLM Model:    {settings.LLM_MODEL}")
print(f"Active Endpoint:     {settings.effective_base_url}")

## 2. Initialize TaskMateAgent

In [ ]:
agent = TaskMateAgent()
print("✅ TaskMateAgent initialized successfully!")
print(f"System prompt length: {len(agent.system_prompt)} characters")
print(f"Max iterations: {agent.max_iterations}")

## 3. Conversational Test (No Tools Required)

In [ ]:
response = agent.process_message(
    user_id="colab_user_123",
    message="Hello! What can you help me with?"
)
print("Response:", response.response)
print("Tool Calls Executed:", len(response.tool_calls))
assert len(response.tool_calls) == 0, "Expected no tool calls for general greeting!"
print("✅ No-tool conversational test passed!")

## 4. Arithmetic Intent (Calculator Tool)

In [ ]:
response = agent.process_message(
    user_id="colab_user_123",
    message="I have 15 lessons to complete over 3 days. How many lessons per day?"
)
print("Response:", response.response)
print("Tool Calls:", response.tool_calls)
print("Tool Results:", response.tool_results)
assert any(tc["name"] == "calculate" for tc in response.tool_calls), "Expected calculate tool call!"
print("✅ Arithmetic intent test passed!")

## 5. Temporal Grounding Intent (Date/Time Tool)

In [ ]:
response = agent.process_message(
    user_id="colab_user_123",
    message="What is the date next Monday?"
)
print("Response:", response.response)
print("Tool Calls:", response.tool_calls)
assert any(tc["name"] == "get_date_time" for tc in response.tool_calls), "Expected get_date_time tool call!"
print("✅ Temporal grounding intent test passed!")